# 01 - PDF Extraction and Text Cache

Three-layer OCR pipeline for 119 sustainability reports (2014-2024).

Layer 1: pdfplumber (native text extraction, digitally generated PDFs)
Layer 2: pdfminer with normalised character decoding (fallback for degraded text)
Layer 3: docTR (image-based OCR for scanned or image-only PDFs)

Page filter: first 90 pages only. Writes cached .txt files to
data/processed/text/<Firm>/<year>.txt and extraction_audit.csv.

In [1]:
%run 00_config.ipynb

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


18:44:45 [INFO] VERIS -- Project root: E:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20
18:44:45 [INFO] VERIS --   [OK] data/raw/reports: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\data\raw\reports
18:44:45 [INFO] VERIS --   [OK] data/processed/text: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\data\processed\text
18:44:45 [INFO] VERIS --   [OK] outputs/csv: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\outputs\csv
18:44:45 [INFO] VERIS --   [OK] outputs/figures: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\outputs\figures
18:44:45 [INFO] VERIS --   [OK] climate_trace/DATA: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\data\raw\climate_trace\DATA
18:44:45 [INFO] VERIS -- 

In [2]:
# Imports for PDF extraction.
import pdfplumber
from pdfminer.high_level import extract_text as pdfminer_extract_text
from pdfminer.layout import LAParams
try:
    from doctr.io import DocumentFile
    from doctr.models import ocr_predictor
    DOCTR_AVAILABLE = True
except ImportError:
    DOCTR_AVAILABLE = False
    log.warning("docTR unavailable; Layer 3 OCR disabled")

In [3]:
# PDF discovery. Maps folder names to canonical firm names via cfg.FIRM_ALIASES,
# so case/space variants (Rio Tinto, Exxon, Conocophillips) all resolve correctly.
def _build_folder_normaliser():
    """Return dict mapping normalised alias to canonical firm name."""
    lookup = {}
    for firm, aliases in cfg.FIRM_ALIASES.items():
        for alias in aliases:
            key = alias.lower().replace(" ", "").replace("_", "").replace("-", "").replace(".", "")
            lookup[key] = firm
        # Also map the canonical name itself (and its lowercase form).
        lookup[firm.lower()] = firm
        lookup[firm.lower().replace(" ", "")] = firm
    return lookup

_FOLDER_TO_FIRM = _build_folder_normaliser()

def discover_pdfs():
    """Return list of (firm, year, pdf_path) tuples for all target firms."""
    records = []
    if not cfg.PDF_DIR.exists():
        log.error(f"PDF directory not found: {cfg.PDF_DIR}")
        return records
    unmatched_folders = []
    for firm_dir in sorted(cfg.PDF_DIR.iterdir()):
        if not firm_dir.is_dir():
            continue
        folder_key = firm_dir.name.lower().replace(" ", "").replace("_", "").replace("-", "").replace(".", "")
        firm = _FOLDER_TO_FIRM.get(folder_key)
        if firm is None:
            unmatched_folders.append(firm_dir.name)
            continue
        for pdf in sorted(firm_dir.glob("*.pdf")):
            stem = pdf.stem
            match = re.search(r"(19|20)\d{2}", stem)
            if not match:
                log.warning(f"No year in filename: {pdf.name}")
                continue
            year = int(match.group(0))
            if year not in cfg.TARGET_YEARS:
                continue
            records.append((firm, year, pdf))
    if unmatched_folders:
        log.warning(f"Unrecognised report folders (ignored): {unmatched_folders}")
    n_firms = len({r[0] for r in records})
    log.info(f"Discovered {len(records)} PDFs across {n_firms} firms")
    return records

In [4]:
# Layer 1: pdfplumber. Fast native-text extraction for digital PDFs.
def extract_layer1(pdf_path, max_pages=None):
    """Return extracted text or empty string on failure."""
    text_parts = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            pages = pdf.pages[:max_pages] if max_pages else pdf.pages
            for page in pages:
                page_text = page.extract_text() or ""
                text_parts.append(page_text)
        return "\n".join(text_parts).strip()
    except Exception as e:
        log.debug(f"Layer 1 failed on {pdf_path.name}: {e}")
        return ""

# Layer 2: pdfminer. Fallback with normalised character decoding.
def extract_layer2(pdf_path, max_pages=None):
    """Return extracted text or empty string on failure."""
    try:
        laparams = LAParams(line_margin=0.5, char_margin=2.0, word_margin=0.1)
        text = pdfminer_extract_text(
            str(pdf_path), laparams=laparams,
            maxpages=max_pages if max_pages else 0,
        )
        return text.strip() if text else ""
    except Exception as e:
        log.debug(f"Layer 2 failed on {pdf_path.name}: {e}")
        return ""

# Layer 3: docTR. OCR for image-only or scanned PDFs.
_doctr_model = None
def _get_doctr():
    global _doctr_model
    if _doctr_model is None and DOCTR_AVAILABLE:
        _doctr_model = ocr_predictor(pretrained=True)
    return _doctr_model

def extract_layer3(pdf_path, max_pages=None):
    """Return extracted text from docTR OCR or empty string on failure."""
    if not DOCTR_AVAILABLE:
        return ""
    try:
        doc = DocumentFile.from_pdf(str(pdf_path))
        if max_pages:
            doc = doc[:max_pages]
        model = _get_doctr()
        result = model(doc)
        json_out = result.export()
        lines = []
        for page in json_out.get("pages", []):
            for block in page.get("blocks", []):
                for line in block.get("lines", []):
                    words = [w["value"] for w in line.get("words", [])]
                    if words:
                        lines.append(" ".join(words))
        return "\n".join(lines).strip()
    except Exception as e:
        log.debug(f"Layer 3 failed on {pdf_path.name}: {e}")
        return ""

In [5]:
# Three-layer routing. Escalates only when previous layer falls below threshold.
def extract_pdf_text(pdf_path, max_pages=None, threshold=None):
    """Run three-layer extraction, return (text, method, chars)."""
    threshold = threshold or cfg.OCR_CHAR_THRESHOLD
    max_pages = max_pages or cfg.PAGE_FILTER_LIMIT

    text = extract_layer1(pdf_path, max_pages)
    if len(text) >= threshold:
        return text, "pdfplumber", len(text)

    text2 = extract_layer2(pdf_path, max_pages)
    if len(text2) > len(text):
        text = text2
    if len(text) >= threshold:
        return text, "pdfminer", len(text)

    text3 = extract_layer3(pdf_path, max_pages)
    if len(text3) > len(text):
        text = text3
        return text, "doctr", len(text)

    return text, "fallback", len(text)

In [6]:
# Run extraction pipeline. Writes text cache + extraction_audit.csv.
# The audit includes both freshly extracted PDFs and any documents already in the
# text cache whose source PDFs are not currently on disk (labelled cached_legacy).
def run_extraction():
    """Extract text from discovered PDFs, then append any cache-only documents."""
    pdfs = discover_pdfs()
    pdf_keys = {(firm, year) for firm, year, _ in pdfs}
    audit_rows = []

    for firm, year, pdf_path in tqdm(pdfs, desc="Extracting PDFs"):
        firm_text_dir = cfg.TEXT_DIR / firm
        firm_text_dir.mkdir(parents=True, exist_ok=True)
        out_path = firm_text_dir / f"{year}.txt"

        if out_path.exists():
            existing = out_path.read_text(encoding="utf-8", errors="ignore")
            audit_rows.append({
                "company": firm, "year": year, "method": "cached",
                "chars": len(existing), "words": len(existing.split()),
                "chunk": 1,
            })
            continue

        text, method, chars = extract_pdf_text(pdf_path)
        out_path.write_text(text, encoding="utf-8")
        audit_rows.append({
            "company": firm, "year": year, "method": method,
            "chars": chars, "words": len(text.split()), "chunk": 1,
        })
        gc.collect()

    # Scan the text cache for documents without a matching raw PDF.
    if cfg.TEXT_DIR.exists():
        for firm_dir in sorted(cfg.TEXT_DIR.iterdir()):
            if not firm_dir.is_dir():
                continue
            firm = firm_dir.name
            if firm not in cfg.ALL_FIRMS:
                continue
            for txt_path in sorted(firm_dir.glob("*.txt")):
                try:
                    year = int(txt_path.stem)
                except ValueError:
                    continue
                if year not in cfg.TARGET_YEARS:
                    continue
                if (firm, year) in pdf_keys:
                    continue
                existing = txt_path.read_text(encoding="utf-8", errors="ignore")
                if len(existing) < 200:
                    continue
                audit_rows.append({
                    "company": firm, "year": year, "method": "cached_legacy",
                    "chars": len(existing), "words": len(existing.split()),
                    "chunk": 1,
                })

    audit = pd.DataFrame(audit_rows).sort_values(["company", "year"]).reset_index(drop=True)
    audit.to_csv(cfg.EXTRACTION_AUDIT_CSV, index=False)
    log.info(f"Extraction audit written: {len(audit)} rows -> {cfg.EXTRACTION_AUDIT_CSV.name}")
    log.info(f"Method breakdown:\n{audit['method'].value_counts().to_string()}")
    log.info(f"Firms in audit: {audit['company'].nunique()}  |  Total words: {audit['words'].sum():,}")
    return audit

audit_df = run_extraction()
display(audit_df.head(15))

18:44:56 [INFO] VERIS -- Discovered 119 PDFs across 12 firms
Extracting PDFs: 100%|██████████| 119/119 [00:00<00:00, 360.93it/s]
18:44:57 [INFO] VERIS -- Extraction audit written: 119 rows -> extraction_audit.csv
18:44:57 [INFO] VERIS -- Method breakdown:
method
cached    119
18:44:57 [INFO] VERIS -- Firms in audit: 12  |  Total words: 4,795,899


,company,year,method,chars,words,chunk
0,BP,2014,cached,218196,34213,1
1,BP,2015,cached,221551,34781,1
2,BP,2016,cached,187712,30117,1
3,BP,2017,cached,151389,24292,1
4,BP,2018,cached,185961,29468,1
5,BP,2019,cached,251342,39253,1
6,BP,2020,cached,350231,56982,1
7,BP,2021,cached,212981,34744,1
8,BP,2022,cached,249755,40829,1
9,BP,2023,cached,267021,41534,1


In [7]:
# ═══════════════════════════════════════════════════════════════════════════
# POST-EXTRACTION SANITY CHECK (optional)
# ═══════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("SANITY CHECK - flag any unexpectedly thin/empty extractions")
print("=" * 70)

issues = []
for _, row in audit_df.iterrows():
    company, year, method, chars, words = row["company"], row["year"], row["method"], row["chars"], row["words"]
    if words == 0 or chars == 0:
        issues.append((company, year, method, words, "🔴 EMPTY - extraction failed silently"))
    elif words < 500:
        issues.append((company, year, method, words, "🟠 VERY THIN - likely broken"))
    elif words < 3000 and method != "cached_legacy":
        issues.append((company, year, method, words, "🟡 THIN - review manually"))
    elif words > 100_000:
        issues.append((company, year, method, words, "🟡 HUGE - 90-page cap may have failed"))

if not issues:
    print("🟢 All 119 extractions are in the expected 3k-100k word range.")
else:
    print(f"⚠️  {len(issues)} flagged rows:")
    print(f"{'Firm':<18} {'Year':>5} {'Method':<20} {'Words':>8}  Status")
    for c, y, m, w, s in issues:
        print(f"{c:<18} {y:>5} {m:<20} {w:>8,}  {s}")

# Totals sanity
print(f"\nCorpus totals:")
print(f"  Rows:         {len(audit_df)}")
print(f"  Firms:        {audit_df['company'].nunique()}")
print(f"  Total words:  {audit_df['words'].sum():,}")
print(f"  Mean words:   {audit_df['words'].mean():,.0f}")
print(f"  Method split:\n{audit_df['method'].value_counts().to_string()}")

SANITY CHECK - flag any unexpectedly thin/empty extractions
🟢 All 119 extractions are in the expected 3k-100k word range.

Corpus totals:
  Rows:         119
  Firms:        12
  Total words:  4,795,899
  Mean words:   40,302
  Method split:
method
cached    119
